# 1a · How many tests are needed to be confident in a displayed IQB score?

**Research question (RQ 1a)** *"Conduct bootstrap resampling to establish minimum measurement thresholds for score reliability."*

In order to display a country's IQB score as a single number(to one decimal), we posit that the score must be **±0.05** of the displayed score with at least **95 %** probability. This follows the guidelines also set forth in Ookla Speedtest Global Index. 


## How many tests? 

IQB score at p50 describes the typical measured connection where half of the speed tests in the country met or exceeded threshold values. Whether that score can be trusted depends on how far the pass rate is from 50%: the closer it is, the more tests are needed to be sure which side of 50% it falls on (i.e. pass or fail). That number grows with the inverse square of the distance from 50%.

Let's take one row from the download data (`downloads_by_country`, November 2025):

|column|value|
|---|---|
| country_code | AD |
| sample_count | 2001 |
| download_p1 | 1.55 |
| download_p5 | 12.08 |
| download_p10 | 22.78 |
| download_p25 | 52.33 |
| download_p50 | 122.22 |
| download_p75 | 286.33 |
| download_p90 | 481.80 |
| download_p95 | 635.13 |
| download_p99 | 859.83 |

Given the threshold of 100 Mbps for download, 100 Mbps falls between `download_p25` (52.33 Mbps) and `download_p50` (122.22 Mbps).

As the data provides only nine percentile values (p1–p99) per country rather than the full cumulative distribution, we construct a continuous CDF by interpolating between the values, which assumes tests are spread evenly between adjacent percentiles.

### Which interpolation scale?

Between two published percentiles, the shape of the distribution is unknown. We compare linear and logarithmic interpolation with a held-out-percentile test, illustrated below for Andorra's download speeds in November 2025.

**Step 1: Hide one percentile.** Each interior percentile is removed in turn and treated as unknown. For Andorra, hiding $P_{25}$ leaves its neighbours $P_{10}$ and $P_{50}$:

| | $P_{10}$ | $P_{25}$ (hidden) | $P_{50}$ |
|---|---|---|---|
| Download (Mbps) | 22.78 | 52.33 | 122.22 |

**Step 2: Predict it from its neighbours.** The hidden value at level $p$ is predicted from the neighbouring values $q_{\text{lo}}$ and $q_{\text{hi}}$ at levels $p_{\text{lo}}$ and $p_{\text{hi}}$. Writing $w = (p - p_{\text{lo}})/(p_{\text{hi}} - p_{\text{lo}})$:

$$
\hat{q}_{\text{linear}} = q_{\text{lo}} + w\,(q_{\text{hi}} - q_{\text{lo}})
$$

$$
\hat{q}_{\text{log}} = \exp\!\big(\ln q_{\text{lo}} + w\,(\ln q_{\text{hi}} - \ln q_{\text{lo}})\big)
$$

For $P_{25}$, $w = (0.25 - 0.10)/(0.50 - 0.10) = 0.375$, giving $\hat{q}_{\text{linear}} \approx 60.1$ Mbps and $\hat{q}_{\text{log}} \approx 42.8$ Mbps, against a true value of 52.33 Mbps.

**Step 3: Score the prediction against each threshold.** Because IQB uses only whether a value passes or fails a threshold, a prediction is scored as correct when it lies on the same side of a threshold as the true value. Every IQB threshold between the two neighbours is checked. For Andorra these are the download thresholds of 25, 50 and 100 Mbps:

| Threshold (Mbps) | True $P_{25}$ (52.3) | Linear (60.1) | Log (42.8) |
|---|---|---|---|
| 25 | above | above ✓ | above ✓ |
| 50 | above | above ✓ | below ✗ |
| 100 | below | below ✓ | below ✓ |


As can be seen in the table above, linear interpolation predicts the p25 as 60.1 Mbps whereas log interpolation predicts the p25 value as 42.8 Mbps. If the threshold to clear for download speed had been 25 Mbps, both linear and logarithmic interpolation values would correctly place the predicted p25 value as above 25 Mbps, and both interpolations therefore "pass" the download speed as meeting the required threshold. However, when taking 50 Mbps as the threshold, linear interpolation predicts 60.1 Mbps. This aggrees with the true p25 value and "passes" the required threshold. Conversely, the log interpolation predicts a p25 value of 42.8 Mbps, thus not clearing the required 50 Mbps for the download threshold. This "fails" the download requirement at p25, though we can see that the true value satisfies the given threshold.  

In this single case linear interpolation is correct for all three thresholds and logarithmic interpolation is correct for two.


**Step 4: Repeat across all cases.** The procedure is repeated for every interior percentile ($P_5$ to $P_{95}$), every country-month, every metric and every threshold between the neighbours. 

In [ ]:
SCALES = {
    "linear": (lambda x: x, lambda y: y),
    "log": (lambda x: np.log(np.maximum(x, 1e-6)), np.exp),
    "log (loss + 0.0001)": (
        lambda x: np.log(x + T.LOSS_OFFSET),
        lambda y: np.exp(y) - T.LOSS_OFFSET,
    ),
}
THRESHOLDS = {
    m: sorted(
        set(fw_high.loc[fw_high.metric == m, "threshold"])
        | set(cfg.loc[cfg.metric == m, "threshold"])
        | set(fw_min.loc[fw_min.metric == m, "threshold"])
    )
    for m in T.METRICS
}
rows = []
for m in T.METRICS:
    for name, (f, finv) in SCALES.items():
        if (m == "loss") != (name == "log (loss + 0.0001)") and name != "linear":
            continue
        hits = cases = 0
        for lo, mid, hi in zip(T.PCTS[:-2], T.PCTS[1:-1], T.PCTS[2:]):
            a, b, actual = (df[f"{m}_p{p}"].to_numpy() for p in (lo, hi, mid))
            pred = finv(f(a) + (mid - lo) / (hi - lo) * (f(b) - f(a)))
            for t in THRESHOLDS[m]:
                between = (np.minimum(a, b) < t) & (np.maximum(a, b) > t)
                hits += ((pred[between] < t) == (actual[between] < t)).sum()
                cases += between.sum()
        rows.append((m, "log" if name != "linear" else "linear", hits / cases, cases))
held = pd.DataFrame(rows, columns=["metric", "scale", "correct side", "cases"]).pivot(
    index="metric", columns="scale", values="correct side"
)
held.loc[list(T.METRICS)].round(3)


The table reports the share of cases in which each scale places the hidden percentile on the correct side of the threshold.

| Metric | Linear | Log |
|---|---|---|
| Download | 0.792 | **0.872** |
| Upload | 0.761 | **0.852** |
| Latency | 0.741 | **0.764** |
| Packet loss | 0.643 | **0.837** |

Logarithmic interpolation is more accurate for every metric, although not in every individual case, as the above example shows. The gain is largest for packet loss, whose distribution is highly skewed. Pass rates are therefore interpolated on a log scale for all four metrics, with an offset of $10^{-4}$ for packet loss, which is often exactly zero. 

Following the held-out test above, we interpolate between adjacent percentiles on a log scale:

$$
F(x) = p_k + \frac{\ln x - \ln q_k}{\ln q_{k+1} - \ln q_k}\,(p_{k+1} - p_k), \qquad q_k \le x \le q_{k+1}
$$

where $q_k$ is the reported value at percentile level $p_k$ (e.g. $q = 52.33$ at $p = 0.25$). This assumes tests are spread evenly in *ratio* between adjacent percentiles: as many tests between 52 and 80 Mbps as between 80 and 122 Mbps.

For Andorra:

$$
F(100) = 0.25 + \frac{\ln 100 - \ln 52.33}{\ln 122.22 - \ln 52.33}\,(0.50 - 0.25) \approx 0.25 + 0.76 \times 0.25 \approx 0.44
$$

So an estimated 44% of tests fall below 100 Mbps, and the pass rate is $\pi(100) = 1 - F(100) \approx 0.56$.

### How to account for standard error? 

Andorra's 2001 tests are a sample. A different set of 2001 tests would give a slightly different pass rate. With $n$ tests, the standard error of the pass rate is

$$
\mathrm{SE} = \sqrt{\frac{\pi(1-\pi)}{n}}
$$

The score is reliable when the pass rate is far enough from 50% that sampling error is unlikely to change the outcome.

Writing $d = \lvert \pi - 0.5 \rvert$ for that distance, we require $d \ge z \cdot \mathrm{SE}$, where $z = 1.96$ for 95% confidence (two-sided).

Solving for $n$:

$$
n_{\text{req}} = \frac{z^2\,\pi(1-\pi)}{d^2}
$$

| Threshold | Pass rate $\pi$ | Distance $d$ | Tests needed | Andorra has |
|---|---|---|---|---|
| 100 Mbps | 0.56 | 0.06 | ~271 | 2001 |

Andorra's `download_p50` (122.22 Mbps) clears 100 Mbps by some margin, so about 271 tests are needed to be 95% confident it is not a chance result and that the score would come out the same if a different set of tests were run.


## How clustered are the tests? 

Tests are not independent: the same users test repeatedly, and a handful of networks may dominate in a given country. $n$ tests therefore carry the information of fewer independent ones. With only monthly percentile tables, the number of independent tests can be estimated from how pass rates move over time. 

We treat each country's monthly pass rate as a slowly moving true level plus independent sampling noise:

$$
\pi_t = \mu_t + \varepsilon_t
$$

Real change in $\mu_t$ tends to persist from one month to the next. Sampling noise $\varepsilon_t$ does not: a month that is too high by chance is followed by one that is, on average, back to normal.

Under this model, the covariance between consecutive changes equals minus the noise variance:

$$
\operatorname{Cov}(\Delta\pi_t,\ \Delta\pi_{t-1}) = -\sigma_\varepsilon^2
$$

Dividing this observed noise variance by the variance independent sampling would produce gives the design effect:

$$
\text{deff} = \frac{\sigma_\varepsilon^2}{\pi(1-\pi)/n}
$$

A design effect of 1 means tests behave as if independent. A value of 2 means the effective sample size is half of `sample_count`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("v2").resolve()))   # iqb_tools lives in v2/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import iqb_tools as T

T.set_style()
cfg = T.config_table()
tests = T.distinct_tests(cfg)
C = T.read_table("country_2025").set_index("country_code")
ctx = T.read_table("country_context").set_index("country_code")
monthly = T.read_table("country_month_scores", parse_dates=["month"])
m25 = monthly[monthly.month.dt.year == 2025]

OUT_TAB = Path("outputs/tables")                 # keep v2/outputs untouched
OUT_TAB.mkdir(parents=True, exist_ok=True)
print(f"Standard: |true − displayed| ≤ {T.TOL} with probability ≥ {T.CONF:.0%}; design effect {T.DEFF}")


In [ ]:
df = T.load_level("country")
tests = T.distinct_tests(cfg)
pr = T.pass_rates(df, tests)
pr[["cc", "month", "n_down", "n_up"]] = df[["country_code", "month", "n_down", "n_up"]]

rows = []
for cc, g in pr.sort_values("month").groupby("cc"):
    g = g.set_index("month").asfreq("MS")  # keep calendar spacing; gaps become NaN
    for k in tests.key:
        x = g[k]
        n = g["n_up"] if k.startswith("upload") else g["n_down"]
        d1 = x.diff()
        lag = (d1 * d1.shift(1)).dropna()
        p = x.mean()
        iid = (x * (1 - x) / n).mean()
        if len(lag) < 12 or not (0.05 < p < 0.95) or not iid > 0:
            continue
        rows.append((cc, k, g.n_down.median(), p, -lag.mean(), iid, x.var()))
dfx = pd.DataFrame(
    rows,
    columns=[
        "country_code",
        "test",
        "n_month",
        "pass_rate",
        "noise_var",
        "iid_var",
        "total_var",
    ],
)
dfx["deff"] = dfx.noise_var / dfx.iid_var
dfx["band"] = pd.cut(
    dfx.n_month,
    [0, 1e3, 1e4, 1e5, np.inf],
    labels=["<1k", "1k–10k", "10k–100k", ">100k"],
)
summary = dfx.groupby("band", observed=True).agg(
    series=("test", "size"),
    countries=("country_code", "nunique"),
    design_effect_median=("deff", "median"),
    design_effect_q25=("deff", lambda s: s.quantile(0.25)),
    design_effect_q75=("deff", lambda s: s.quantile(0.75)),
    estimate_negative=("deff", lambda s: (s < 0).mean()),
)
summary["independent-sampling share of monthly variance"] = dfx.groupby(
    "band", observed=True
).apply(lambda x: (x.iid_var / x.total_var).median(), include_groups=False)
print(f"{len(dfx):,} monthly series, {dfx.country_code.nunique()} countries")
summary.round(2)


In [ ]:
identifiable = dfx[dfx.n_month < 10_000]
deff_hat = identifiable.deff.median()
print(
    f"Design effect where sampling noise is visible (< 10,000 tests a month): median {deff_hat:.1f} "
    f"(IQR {identifiable.deff.quantile(0.25):.1f}–{identifiable.deff.quantile(0.75):.1f}, {len(identifiable)} series)"
)
print(f"Value used in iqb_tools.DEFF: {T.DEFF}")
print("\nBy threshold (series with < 10,000 tests a month):")
print(identifiable.groupby("test").deff.median().round(1).to_string())

# Pooling where a reporting period falls short

A country below the minimum in a month can still get a score by pooling months. Pass rates combine exactly across months as a test-weighted average (notebook 01), so a pooled score is simply the score of all tests in the window.
